[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KP-365/Fake_news/blob/main/eval_faithfulness.ipynb)

# Human faithfulness review of generated explanations

This notebook samples 10 committed escalation rows, generates a fresh explanation from displayed structured signals, and creates a CSV template for human review. Use a GPU runtime.

> **Scope note:** `evaluation/escalation_results.csv` preserves labels and NLI verdicts, but not classifier confidence, MC uncertainty, NLI scores, or evidence counts. The notebook therefore recomputes one coherent live signal set from each sampled text snippet and attaches it to that sampled row before calling `explain_decision()`. DDG evidence is live, so these signals are a new faithfulness-review run rather than a reproduction of the historical escalation run.

## 1. Clone the repository and install its pinned environment

The setup is safe to rerun in one Colab session. It installs the repository's versioned requirements rather than selecting package versions in this notebook.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/KP-365/Fake_news.git"
REPO_DIR = Path("/content/Fake_news")

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print(f"Ready in {Path.cwd()}")

Ready in /content/Fake_news


## 2. Load the escalation results and select 10 rows

`row_id` is the zero-based row position in the committed CSV. Sampling is without replacement and uses NumPy's generator with seed 42.

In [2]:
import numpy as np
import pandas as pd

RESULTS_PATH = Path("evaluation/escalation_results.csv")
REQUIRED_COLUMNS = {
    "text_snippet",
    "true_label",
    "classifier_label",
    "verdict",
    "final_label",
}

escalation_results = pd.read_csv(RESULTS_PATH)
missing_columns = REQUIRED_COLUMNS.difference(escalation_results.columns)
if missing_columns:
    raise ValueError(f"Missing escalation columns: {sorted(missing_columns)}")
if len(escalation_results) < 10:
    raise ValueError("At least 10 escalation rows are required")

rng = np.random.default_rng(42)
sample_indices = rng.choice(
    escalation_results.index.to_numpy(), size=10, replace=False
)
sampled_rows = escalation_results.loc[sample_indices].copy()
sampled_rows.insert(0, "row_id", sampled_rows.index.astype(int))
sampled_rows = sampled_rows.reset_index(drop=True)

print("Selected source row IDs:", sampled_rows["row_id"].tolist())
display(
    sampled_rows[[
        "row_id", "text_snippet", "true_label",
        "classifier_label", "verdict", "final_label"
    ]]
)

Selected source row IDs: [96, 71, 8, 60, 41, 94, 68, 9, 19, 82]


,row_id,text_snippet,true_label,classifier_label,verdict,final_label
0,96,Hillary Clinton KNEW 5 years ago Anthony Weine...,fake,fake,insufficient,fake
1,71,Hustle at Hofstra With Donald Trump and Hillar...,fake,real,supported,real
2,8,Donald Trump revokes Washington Post press acc...,real,fake,refuted,fake
3,60,Clinton ‘Not Concerned’ About New Flap Over Cl...,real,real,refuted,fake
4,41,"Blood money, killer cops: How privatization is...",real,real,supported,real
5,94,“Not A ‘Real’ Union”: Emails Show Clinton Camp...,fake,fake,supported,real
6,68,This Is the Least Important Election of Our Li...,real,real,insufficient,real
7,9,"This man wants to become president, pass one l...",real,fake,supported,real
8,19,PressTV-Hezbollah shares Aoun’s victory: Leban...,fake,fake,insufficient,fake
9,82,Goldman Sachs Endorses Hillary Clinton For Pre...,fake,fake,refuted,fake


## 3. Generate and display explanations

The Anthropic key is read with `getpass()`, passed directly to each request, never written to a DataFrame, file, environment variable, or notebook output, and deleted from the notebook namespace in `finally`. For each source row, the exact structured signal dictionary is printed immediately before its explanation.

This cell performs live DDG retrieval and model inference, so outputs and runtime can vary.

In [4]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [5]:
from getpass import getpass
import json
import time

from explain import explain_decision
from pipeline import classify_with_uncertainty
from predict import load_model
from verify import verify_claim

model, tokenizer, device = load_model()
print(f"Classifier startup device: {device}")

def compute_structured_signals(text: str) -> dict:
    classifier_label, confidence, mc_uncertainty = classify_with_uncertainty(
        text, model=model, tokenizer=tokenizer, device=device
    )
    verification = verify_claim(text)
    return {
        "classifier_label": classifier_label,
        "confidence": float(confidence),
        "mc_uncertainty": float(mc_uncertainty),
        "nli_verdict": verification.get("verdict", "insufficient"),
        "max_entailment": float(verification.get("max_entailment", 0.0) or 0.0),
        "max_contradiction": float(verification.get("max_contradiction", 0.0) or 0.0),
        "evidence_count": int(len(verification.get("evidence", []))),
    }

anthropic_key = getpass("Anthropic API key (hidden; held in memory only): " ).strip()
if not anthropic_key:
    raise ValueError("An Anthropic API key is required for this review")

faithfulness_records = []
try:
    for position, row in sampled_rows.iterrows():
        signals = compute_structured_signals(row["text_snippet"])
        explanation = explain_decision(**signals, api_key=anthropic_key)
        record = {
            "row_id": int(row["row_id"]),
            "signals": signals,
            "explanation": explanation,
        }
        faithfulness_records.append(record)

        print(f"\n=== Source row {record['row_id']} ===")
        print("Structured signals passed to explain_decision():")
        print(json.dumps(signals, indent=2))
        print("Generated explanation:")
        print(explanation)

        if position < len(sampled_rows) - 1:
            time.sleep(2)
finally:
    anthropic_key = ""
    del anthropic_key

print(f"\nGenerated {len(faithfulness_records)} explanations.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Classifier startup device: cuda
Anthropic API key (hidden; held in memory only): ··········
Retrieving evidence...
Retrieved 5 evidence result(s).


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Source row 96 ===
Structured signals passed to explain_decision():
{
  "classifier_label": "real",
  "confidence": 0.6691640019416809,
  "mc_uncertainty": 0.14274297654628754,
  "nli_verdict": "insufficient",
  "max_entailment": 0.021514892578125,
  "max_contradiction": 0.01024627685546875,
  "evidence_count": 5
}
Generated explanation:
The system classified this article as real with moderate confidence of 0.67, supported by relatively low model uncertainty of 0.14, indicating the classifier made a fairly stable prediction. However, the NLI analysis found insufficient evidence to support or refute the claim, with both entailment and contradiction scores near zero across five pieces of evidence. While the NLI verdict cannot resolve the label through logical inference, this does not change the classifier's real designation, which was determined through other signal patterns in the article.
Retrieving evidence...
Retrieved 5 evidence result(s).

=== Source row 71 ===
Structured signa

## 4. Create the manual review table

The generated explanation is copied into `claim_in_explanation` without automatic interpretation. Review it against the displayed signals, split it into additional claim-level rows when needed, enter `yes` or `no`, and add notes. Rerun the save line after manual edits.

In [6]:
REVIEW_PATH = Path("evaluation/faithfulness_review.csv")

review_table = pd.DataFrame(
    {
        "row_id": [record["row_id"] for record in faithfulness_records],
        "claim_in_explanation": [
            record["explanation"] for record in faithfulness_records
        ],
        "matches_signals_yes_no": "",
        "notes": "",
    }
)

REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
review_table.to_csv(REVIEW_PATH, index=False)
display(review_table)
print(f"Saved manual review template to {REVIEW_PATH}")

,row_id,claim_in_explanation,matches_signals_yes_no,notes
0,96,The system classified this article as real wit...,,
1,71,The system classified this article as real wit...,,
2,8,The system classified this article as fake wit...,,
3,60,The system classified this article as fake wit...,,
4,41,The system classified this article as real wit...,,
5,94,The system classified this content as fake wit...,,
6,68,The system classified this content as fake wit...,,
7,9,The system classified this article as fake wit...,,
8,19,The system classified this article as real wit...,,
9,82,The system classified this content as fake wit...,,


Saved manual review template to evaluation/faithfulness_review.csv
